# D2 · M2.3 + M2.4 — Hybrid Search, Retrieval & Re-ranking

**Meridian Bank, support assistant.** Priya from support reported one bug: *"Customers ask about
missing salary. The bot answers with account benefits."* This notebook finds the cause and ships
the fix.

**Worked side by side with the concept page.** Four round trips today:

| You read on the page | You run here |
|---|---|
| Concepts 1–3 (keyword, semantic) | **Part 1** — two engines, same articles |
| Concepts 4–6 (hybrid, fusion) | **Part 2** — merge the two lists |
| Concepts 7–8 (retriever, re-ranking) | **Part 3** — retrieve wide, re-rank narrow |
| Concept 9 (full pipeline) | **Part 4** — end to end, with a real metric |

**This notebook is fully working code — nothing to fill in.** Run it top to bottom. Every part
prints something you are meant to *look at*, and each markdown cell tells you what you should see.

## Setup — run these two cells once

The corpus is 40 real Meridian help articles with metadata (product, segment, region, updated date).
Eight of them are the ones you saw on the concept page.

In [ ]:
# Run once. "Requirement already satisfied" is a good outcome.
%pip install -q rank_bm25 chromadb

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi

# The .env sits next to this notebook (or anywhere up the tree). Without this line
# os.environ will not see your key, even though the file exists.
# Find the .env file wherever it lives: this folder, any folder above it, or the
# Day1/labs/starter/ folder SETUP.md originally told you to create it in. One key
# file for the whole repo -- you never need to copy it between Day folders.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)

client = OpenAI()
MODEL_EMBED = "text-embedding-3-small"
MODEL_CHAT = "gpt-4o-mini"


def find_data_dir():
    """Walk up from wherever the kernel started until we find Day2/data."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        hit = candidate / "Day2" / "data"
        if hit.is_dir():
            return hit
        if (candidate / "data" / "D2_M2.3_meridian_articles.json").is_file():
            return candidate / "data"
    raise FileNotFoundError("Could not locate Day2/data -- open this notebook from inside the student repo.")


DATA = find_data_dir()
with open(DATA / "D2_M2.3_meridian_articles.json", encoding="utf-8") as f:
    ARTICLES = json.load(f)
with open(DATA / "D2_M2.3_eval_queries.json", encoding="utf-8") as f:
    EVAL = json.load(f)

BY_ID = {a["id"]: a for a in ARTICLES}
QUERY = "My salary hasn't arrived."   # Ravi's message. We use it all the way through.

print(f"Loaded {len(ARTICLES)} Meridian articles and {len(EVAL)} evaluation queries.")
print(f"Data folder: {DATA}")
print()
for a in ARTICLES[:8]:
    print(f"  id {a['id']:>2} | {a['product']:<8} | {a['segment']:<9} | {a['region']} | {a['title']}")
print("  ... 32 more")

---
# Part 1 — Two engines over the same articles

> 📖 **Read Concepts 1–3 on the concept page first** (Why search is hard → Keyword search →
> Semantic search). Then run the three cells below and come back to the page for Concept 4.

Same 40 articles. Two completely different ways to search them.

In [ ]:
# ---- Engine 1: keyword search (BM25) --------------------------------------
# BM25 counts words. Rare words score more than common ones. Repetition adds up.

def tokenize(text):
    return re.findall(r"[a-z0-9\-]+", text.lower())


corpus_tokens = [tokenize(a["title"] + " " + a["text"]) for a in ARTICLES]
bm25 = BM25Okapi(corpus_tokens)


def keyword_scores(query):
    """Returns {article_id: bm25_score} for every article."""
    scores = bm25.get_scores(tokenize(query))
    return {a["id"]: float(s) for a, s in zip(ARTICLES, scores)}


def show(title, ranked, gold_id=None, n=5):
    """ranked = list of (article_id, score), already sorted best first."""
    print(title)
    for rank, (aid, score) in enumerate(ranked[:n], start=1):
        mark = "  <-- correct answer" if gold_id and aid == gold_id else ""
        print(f"  {rank}. [{score:6.3f}] id {aid:>2}  {BY_ID[aid]['title'][:56]}{mark}")
    print()


kw = keyword_scores(QUERY)
kw_ranked = sorted(kw.items(), key=lambda kv: -kv[1])

print(f"QUERY: {QUERY}\n")
show("KEYWORD SEARCH (BM25) -- top 5", kw_ranked, gold_id=1)

**Look at rank 1.** *"Salary account benefits and zero-balance rules"* — a page about minimum
balances, which answers nothing. It wins because it says the word "salary" six times.

That is Priya's bug, reproduced in one cell. The article that actually answers Ravi is at **rank 4**.

In [ ]:
# ---- Engine 2: semantic search (embeddings + Chroma) -----------------------
import chromadb

texts = [a["title"] + ". " + a["text"] for a in ARTICLES]

# One batched call embeds all 40 articles at once -- cheaper and much faster than 40 calls.
resp = client.embeddings.create(model=MODEL_EMBED, input=texts)
doc_vectors = [d.embedding for d in resp.data]

chroma = chromadb.Client()
try:
    chroma.delete_collection("meridian_hybrid")   # so re-running this cell is safe
except Exception:
    pass
collection = chroma.create_collection(
    name="meridian_hybrid",
    metadata={"hnsw:space": "cosine"},            # so distance = 1 - cosine similarity
)
collection.add(
    ids=[str(a["id"]) for a in ARTICLES],
    embeddings=doc_vectors,
    documents=texts,
    metadatas=[{"product": a["product"], "segment": a["segment"],
                "region": a["region"], "updated": a["updated"]} for a in ARTICLES],
)


def semantic_scores(query, where=None):
    """Returns {article_id: cosine_similarity} for every article that passes the filter."""
    qv = client.embeddings.create(model=MODEL_EMBED, input=query).data[0].embedding
    res = collection.query(
        query_embeddings=[qv],
        n_results=len(ARTICLES),
        where=where,
    )
    ids = [int(i) for i in res["ids"][0]]
    sims = [1.0 - d for d in res["distances"][0]]   # cosine distance -> similarity
    return dict(zip(ids, sims))


sem = semantic_scores(QUERY)
sem_ranked = sorted(sem.items(), key=lambda kv: -kv[1])

show("SEMANTIC SEARCH (embeddings) -- top 5", sem_ranked, gold_id=1)

In [ ]:
# ---- Side by side ---------------------------------------------------------
print(f"QUERY: {QUERY}\n")
print(f"{'#':<3} {'KEYWORD (BM25)':<48} | SEMANTIC (embeddings)")
print("-" * 100)
for i in range(5):
    k_id, k_s = kw_ranked[i]
    s_id, s_s = sem_ranked[i]
    left = f"{BY_ID[k_id]['title'][:38]:<38} {k_s:5.2f}"
    right = f"{BY_ID[s_id]['title'][:38]:<38} {s_s:5.2f}"
    print(f"{i+1:<3} {left:<48} | {right}")

print()
print(f"Correct answer (id 1) sits at keyword rank "
      f"{[a for a, _ in kw_ranked].index(1) + 1}, semantic rank "
      f"{[a for a, _ in sem_ranked].index(1) + 1}.")

### What you should see

- **Keyword** put *"Salary account benefits"* first. Wrong — it just repeats the word "salary".
- **Semantic** did better and often puts id 1 at or near the top, but *"Payment not received"* is
  right behind it. Meaning-matching cannot tell that **NEFT** is the decisive word.
- Neither engine is reliable on its own. That is the whole argument for hybrid search.

> 🔵 **Go back to the concept page** — read Concepts 4, 5 and 6 (Where each one fails → Hybrid
> search → How fusion works), then return here for Part 2.

---
# Part 2 — Fusion: merging the two lists

Two ranked lists, one final ranking. Two ways to do it, and you will run both.

In [ ]:
# ---- Way 1: weighted score fusion -----------------------------------------
# BM25 scores are unbounded (0 to ~8 here). Cosine similarity is roughly 0 to 1.
# You cannot add them directly, so normalise both to 0-1 first. This step is the
# one people forget, and it silently ruins the ranking.

def normalise(scores):
    lo, hi = min(scores.values()), max(scores.values())
    span = hi - lo or 1.0
    return {k: (v - lo) / span for k, v in scores.items()}


def weighted_fusion(kw_scores, sem_scores, alpha=0.5):
    """alpha = how much to trust semantic. (1 - alpha) goes to keyword."""
    k, s = normalise(kw_scores), normalise(sem_scores)
    fused = {aid: alpha * s[aid] + (1 - alpha) * k[aid] for aid in s}
    return sorted(fused.items(), key=lambda kv: -kv[1])


for alpha in (0.0, 0.2, 0.5, 0.8, 1.0):
    ranked = weighted_fusion(kw, sem, alpha)
    pos = [a for a, _ in ranked].index(1) + 1
    top = BY_ID[ranked[0][0]]["title"][:46]
    print(f"alpha={alpha:.1f} (semantic weight)  ->  rank 1: {top:<48} | correct answer at rank {pos}")

**Read the last column.** At `alpha=0.0` (pure keyword) the correct article is buried. At
`alpha=1.0` (pure semantic) it wins, but only just. Anywhere in the middle it is stable at rank 1 —
which is exactly the flat, boring behaviour you want in production. You tune alpha on an eval set,
never by feel.

In [ ]:
# ---- Way 2: Reciprocal Rank Fusion (RRF) ----------------------------------
# Ignores the scores completely. Uses only the POSITION in each list.
# score = sum over lists of 1 / (k + rank). k = 60 is the standard constant.

def rrf_fusion(list_of_ranked_lists, k=60, tiebreak=None):
    fused = {}
    for ranked in list_of_ranked_lists:
        for rank, (aid, _score) in enumerate(ranked, start=1):
            fused[aid] = fused.get(aid, 0.0) + 1.0 / (k + rank)
    # Exact ties are common with RRF (two docs can land on the same pair of ranks).
    # Break them with the raw semantic score so the output is stable between runs.
    tiebreak = tiebreak or {}
    return sorted(fused.items(), key=lambda kv: (-kv[1], -tiebreak.get(kv[0], 0.0)))


rrf_ranked = rrf_fusion([kw_ranked, sem_ranked], tiebreak=sem)

# Show the arithmetic for the top few, so the formula stops being abstract
kw_pos = {aid: i + 1 for i, (aid, _) in enumerate(kw_ranked)}
sem_pos = {aid: i + 1 for i, (aid, _) in enumerate(sem_ranked)}

print(f"QUERY: {QUERY}\n")
print(f"{'#':<3} {'article':<44} {'kw rank':>8} {'sem rank':>9} {'RRF score':>11}")
print("-" * 80)
for i, (aid, score) in enumerate(rrf_ranked[:6], start=1):
    mark = "  <-- correct" if aid == 1 else ""
    print(f"{i:<3} {BY_ID[aid]['title'][:44]:<44} {kw_pos[aid]:>8} {sem_pos[aid]:>9} {score:>11.5f}{mark}")

**Notice the trade.** A document that both engines placed reasonably beats a document that one
engine loved and the other ignored. That is the whole point of the `+ 60`: it flattens the gap
between rank 1 and rank 2, so *agreement* matters more than one engine shouting loudly.

Elasticsearch, OpenSearch, Azure AI Search and Qdrant all ship this exact formula as their
built-in hybrid mode.

In [ ]:
# ---- All four rankings, one table -----------------------------------------
w_ranked = weighted_fusion(kw, sem, alpha=0.5)
columns = [("KEYWORD", kw_ranked), ("SEMANTIC", sem_ranked),
           ("WEIGHTED a=0.5", w_ranked), ("RRF", rrf_ranked)]

print(f"QUERY: {QUERY}\n")
header = "".join(f"{name:<26}" for name, _ in columns)
print(f"{'#':<3}{header}")
print("-" * (3 + 26 * len(columns)))
for i in range(5):
    row = ""
    for _, ranked in columns:
        aid = ranked[i][0]
        label = f"{'* ' if aid == 1 else ''}{BY_ID[aid]['title'][:22]}"
        row += f"{label:<26}"
    print(f"{i+1:<3}{row}")
print("\n* = the article that actually answers the question")

### What you should see

The correct NEFT salary article climbs to **rank 1 or 2 under both fusion methods**, from rank 4 on
keyword alone. (Embedding runs vary slightly — what matters is the movement, and that the useless
"benefits" page is no longer first.)

You have just fixed Priya's bug — for one query. Part 3 makes it hold up at scale.

> 🔵 **Go back to the concept page** — read Concepts 7 and 8 (The retriever → Re-ranking), then
> return here for Part 3.

---
# Part 3 — Retrieve wide, re-rank narrow

Two ideas in this part:

1. **Metadata filters** shrink the pool *before* any scoring happens.
2. **Re-ranking** takes 20 rough candidates and picks the best 5 properly.

In [ ]:
# ---- Metadata filter: delete the wrong answers before you score anything ---
# Ravi is a retail customer in India. Corporate and US articles cannot possibly
# be right -- so do not let them compete.

CUSTOMER = {"segment": "retail", "region": "IN"}
WHERE = {"$and": [{"segment": {"$eq": CUSTOMER["segment"]}},
                  {"region":  {"$eq": CUSTOMER["region"]}}]}

sem_all = semantic_scores(QUERY)
sem_filtered = semantic_scores(QUERY, where=WHERE)

print(f"Pool without filter: {len(sem_all)} articles")
print(f"Pool with  filter:   {len(sem_filtered)} articles  (segment=retail, region=IN)")
print()
print("Articles the filter removed from the competition entirely:")
for aid in sorted(set(sem_all) - set(sem_filtered)):
    a = BY_ID[aid]
    print(f"  id {aid:>2} | {a['segment']:<9} | {a['region']} | {a['title'][:52]}")

Look at **id 38, "Direct deposit not received - US payroll"**. Semantically it is an extremely close
match to *"my salary hasn't arrived"* — and it is completely wrong for an Indian retail customer.
No embedding model will ever fix that. A one-line filter does.

In [ ]:
# ---- Retrieve 20 candidates with hybrid search ----------------------------
# Deliberately more than we will show the LLM. If the right article is not in
# this list, nothing downstream can rescue it.

K_RETRIEVE = 20
K_FINAL = 5


def hybrid_retrieve(query, k=K_RETRIEVE, where=WHERE):
    t0 = time.perf_counter()
    s = semantic_scores(query, where=where)
    allowed = set(s)
    k_all = keyword_scores(query)
    kscores = {aid: v for aid, v in k_all.items() if aid in allowed}   # same filter, both engines
    ranked = rrf_fusion([
        sorted(kscores.items(), key=lambda kv: -kv[1]),
        sorted(s.items(), key=lambda kv: -kv[1]),
    ], tiebreak=s)
    elapsed = (time.perf_counter() - t0) * 1000
    return [aid for aid, _ in ranked[:k]], elapsed


candidates, retrieve_ms = hybrid_retrieve(QUERY)
print(f"Retrieved {len(candidates)} candidates in {retrieve_ms:.0f} ms\n")
for rank, aid in enumerate(candidates, start=1):
    mark = "  <-- correct answer" if aid == 1 else ""
    print(f"  {rank:>2}. id {aid:>2}  {BY_ID[aid]['title'][:56]}{mark}")

In [ ]:
# ---- Re-rank with an LLM --------------------------------------------------
# A cross-encoder reads the query and the document TOGETHER, then scores relevance.
# Here we use gpt-4o-mini as the cross-encoder: one call scores all 20 candidates.
#
# HONEST NOTE ON SPEED: a hosted cross-encoder (Cohere Rerank, or a local BGE model)
# does this in ~100 ms. An LLM takes 1-3 seconds. We use the LLM because you already
# have the key and there is nothing to install. The RANKING BEHAVIOUR is what matters
# here -- production teams use a real cross-encoder for the latency.

RERANK_PROMPT = """You are a relevance scorer for a bank's support search.
Score how well each passage ANSWERS the customer's question, from 0 to 10.
A passage that merely mentions the same words but answers nothing scores low.

Customer question: {query}

Passages:
{passages}

Return JSON only, in this exact shape: {{"scores": [{{"id": <int>, "score": <number>}}, ...]}}
Include every passage id exactly once."""


def llm_rerank(query, candidate_ids, top_k=K_FINAL):
    passages = "\n".join(
        f"[{aid}] {BY_ID[aid]['title']}. {BY_ID[aid]['text'][:300]}" for aid in candidate_ids
    )
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL_CHAT,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user",
                   "content": RERANK_PROMPT.format(query=query, passages=passages)}],
    )
    elapsed = (time.perf_counter() - t0) * 1000
    scored = json.loads(resp.choices[0].message.content)["scores"]
    keep = {int(s["id"]): float(s["score"]) for s in scored if int(s["id"]) in set(candidate_ids)}
    ranked = sorted(keep.items(), key=lambda kv: -kv[1])
    return ranked[:top_k], elapsed


reranked, rerank_ms = llm_rerank(QUERY, candidates)

print(f"QUERY: {QUERY}\n")
print(f"{'':<3}{'BEFORE re-ranking (hybrid top 5)':<50}{'AFTER re-ranking (top 5)'}")
print("-" * 100)
for i in range(K_FINAL):
    before = BY_ID[candidates[i]]["title"][:44]
    aid, score = reranked[i]
    after = f"{BY_ID[aid]['title'][:38]} ({score:.1f})"
    star = "*" if aid == 1 else " "
    print(f"{i+1:<3}{before:<50}{star}{after}")

moved = candidates.index(reranked[0][0]) + 1
print(f"\nRetrieval: {retrieve_ms:6.0f} ms for {len(candidates)} candidates")
print(f"Re-ranking:{rerank_ms:6.0f} ms for {len(candidates)} candidates -> top {K_FINAL}")
print(f"The new rank 1 was sitting at position {moved} before re-ranking.")

In [ ]:
# ---- Does it hold up across different phrasings? --------------------------
for q in ["My paycheck is missing.",
          "I think my employer transferred the money but I don't see it.",
          "Status of TXN-99213?"]:
    cands, _ = hybrid_retrieve(q)
    top, _ = llm_rerank(q, cands, top_k=3)
    print(f"\nQ: {q}")
    print(f"   hybrid rank 1  : {BY_ID[cands[0]]['title'][:58]}")
    print(f"   after re-rank  : {BY_ID[top[0][0]]['title'][:58]}")

### What you should see

- At least one query where the re-ranker **promotes an article from deep in the list to rank 1**.
- `"Status of TXN-99213?"` works because the keyword half of hybrid matched the exact string.
  Semantic search alone had no idea what that code was.
- Re-ranking costs real time. It is still the single biggest accuracy lever in the pipeline.

> 🔵 **Go back to the concept page** — read Concept 9 (The full retrieval pipeline), then return
> here for Part 4.

---
# Part 4 — End to end, and a number you can send to Priya

Everything so far produced *documents*. Now the LLM turns them into an *answer* — and then we
measure whether any of this actually worked.

In [ ]:
# ---- The full pipeline in one function ------------------------------------
ANSWER_PROMPT = """You are Meridian Bank's support assistant.
Answer the customer using ONLY the context below. If the context does not contain
the answer, say you don't have that information and offer to connect a human.
Keep it to three sentences. End with the article title you used.

Context:
{context}

Customer: {question}"""


def answer(question, verbose=False):
    cands, t_ret = hybrid_retrieve(question)          # 1. filter + hybrid retrieve 20
    top, t_rr = llm_rerank(question, cands)           # 2. re-rank down to 5
    context = "\n\n".join(
        f"[{aid}] {BY_ID[aid]['title']}. {BY_ID[aid]['text']}" for aid, _ in top
    )                                                  # 3. prompt builder
    t0 = time.perf_counter()
    resp = client.chat.completions.create(             # 4. LLM
        model=MODEL_CHAT, temperature=0,
        messages=[{"role": "user",
                   "content": ANSWER_PROMPT.format(context=context, question=question)}],
    )
    t_llm = (time.perf_counter() - t0) * 1000
    if verbose:
        print(f"   [retrieve {t_ret:.0f} ms | rerank {t_rr:.0f} ms | generate {t_llm:.0f} ms]")
    return resp.choices[0].message.content.strip(), [aid for aid, _ in top]


ANSWER_LOG = []

for q in ["My salary hasn't arrived.",
          "My paycheck is missing.",
          "NEFT transfer not received.",
          "Salary credit is delayed again this month.",
          "I think my employer transferred the money but I don't see it."]:
    text, used = answer(q, verbose=True)
    ANSWER_LOG.append({"query": q, "answer": text, "source_ids": used})
    print(f"\nCUSTOMER: {q}")
    print(f"BOT     : {text}")
    print(f"SOURCES : {used}")
    print("-" * 90)

Five different sentences. Same underlying problem. Same correct article behind every answer —
which is exactly what Priya asked for.

In [ ]:
# ---- Measure it. "It feels better" is not a metric. -----------------------
# Recall@k = in what fraction of queries was the correct article in the top k
# documents the LLM actually received?

def evaluate_semantic_only():
    hits1 = hits5 = 0
    for row in EVAL:
        ranked = sorted(semantic_scores(row["query"]).items(), key=lambda kv: -kv[1])
        top_ids = [aid for aid, _ in ranked[:5]]
        hits1 += row["gold_id"] == top_ids[0]
        hits5 += row["gold_id"] in top_ids
    return hits1 / len(EVAL), hits5 / len(EVAL)


def evaluate_hybrid_reranked():
    hits1 = hits5 = 0
    for row in EVAL:
        cands, _ = hybrid_retrieve(row["query"])
        top, _ = llm_rerank(row["query"], cands)
        top_ids = [aid for aid, _ in top]
        hits1 += row["gold_id"] == top_ids[0]
        hits5 += row["gold_id"] in top_ids
    return hits1 / len(EVAL), hits5 / len(EVAL)


print("Running 10 evaluation queries through both pipelines. This takes about a minute.\n")
base1, base5 = evaluate_semantic_only()
new1, new5 = evaluate_hybrid_reranked()

print(f"{'pipeline':<24}{'Recall@1':>10}{'Recall@5':>10}")
print("-" * 44)
print(f"{'semantic only':<24}{base1:>10.0%}{base5:>10.0%}")
print(f"{'hybrid + re-ranking':<24}{new1:>10.0%}{new5:>10.0%}")
print(f"{'change':<24}{new1 - base1:>+10.0%}{new5 - base5:>+10.0%}")
print()
print("Recall@1 is the honest number -- it asks whether the BEST document was first,")
print("which is what decides the answer the customer actually reads.")


In [ ]:
# ---- The status email -----------------------------------------------------
print(f"""
To: Priya, Head of Support
Subject: Salary query bug -- fixed and measured

Cause: retrieval, not the model. Keyword-style matching ranked 'Salary account benefits'
first because it repeats the word 'salary'. Semantic-only retrieval found the right
neighbourhood but could not weight the term 'NEFT'.

Fix shipped:
  1. Hybrid retrieval (BM25 + embeddings, merged with reciprocal rank fusion)
  2. Metadata filter on customer segment and region, applied before scoring
  3. LLM re-ranking of 20 candidates down to the 5 sent to the model

Measured on our 10-query evaluation set:
  Semantic only        Recall@1 = {base1:.0%}   Recall@5 = {base5:.0%}
  Hybrid + re-ranking  Recall@1 = {new1:.0%}   Recall@5 = {new5:.0%}

All five phrasings of the missing-salary question now resolve to the correct article.
""")

---
## What you built

| Stage | What it does | Why it is there |
|---|---|---|
| Metadata filter | Cuts 40 articles to the ones this customer could possibly need | Deletes whole classes of wrong answers, for free |
| BM25 keyword search | Exact word and ID matching | Catches `TXN-99213`, `NEFT`, product names |
| Vector search | Meaning matching | Catches "paycheck" when the article says "salary" |
| RRF fusion | Merges the two ranked lists | Agreement between engines beats one loud engine |
| Re-ranker | Reads query + document together, keeps the best 5 | The right answer is often below rank 5 |
| Prompt builder | Documents become text in the prompt | The LLM sees nothing else — no vector store access |
| Evaluation | Recall@5 on a fixed query set | The only way to know a change helped |

**Next module (M2.5):** this whole pipeline collapses into a LangChain retrieval chain, and you
add conversation memory on top.

### Grounding check — does it refuse when it should?

A support bot that invents an answer is worse than one that says "I don't know". The prompt tells
the model to answer **only** from the retrieved context. Here is a question Meridian's knowledge
base genuinely cannot answer.

In [ ]:
# ---- Out-of-scope question: the pipeline should decline, not invent ------
OUT_OF_SCOPE = "What is Meridian Bank's share price today?"

refusal_text, refusal_sources = answer(OUT_OF_SCOPE)
print(f"CUSTOMER: {OUT_OF_SCOPE}")
print(f"BOT     : {refusal_text}")
print(f"SOURCES : {refusal_sources}")

# A crude but honest check: did it decline rather than state a number?
declined = any(p in refusal_text.lower() for p in
               ["don't have", "do not have", "not have that information", "cannot",
                "can't", "unable", "connect you", "human"])
print(f"\nDeclined instead of inventing an answer: {declined}")

**Why this matters more than it looks.** Retrieval still returned five documents — it always does.
Nothing in the retrieval stack knows the question is unanswerable. The *only* thing standing
between the customer and an invented share price is that one line in the prompt telling the model
to answer solely from context. Test it every time you change the prompt.

---
## Save your results

This writes `m2_3_m2_4_results.json` next to the notebook. The autograder reads that file — it does
not re-run your notebook. Run this cell last, after every cell above has run successfully.

In [ ]:
# ---- Save results for the autograder --------------------------------------
results = {
    "query": QUERY,
    "keyword_top5": [aid for aid, _ in kw_ranked[:5]],
    "semantic_top5": [aid for aid, _ in sem_ranked[:5]],
    "rrf_top5": [aid for aid, _ in rrf_ranked[:5]],
    "weighted_top5": [aid for aid, _ in weighted_fusion(kw, sem, alpha=0.5)[:5]],
    "filter": {
        "customer": CUSTOMER,
        "pool_before": len(sem_all),
        "pool_after": len(sem_filtered),
        "removed_ids": sorted(set(sem_all) - set(sem_filtered)),
    },
    "retrieval": {
        "k_retrieve": K_RETRIEVE,
        "k_final": K_FINAL,
        "candidate_ids": candidates,
        "reranked_ids": [aid for aid, _ in reranked],
        "retrieve_ms": round(retrieve_ms, 1),
        "rerank_ms": round(rerank_ms, 1),
    },
    "answers": ANSWER_LOG,
    "grounding": {
        "question": OUT_OF_SCOPE,
        "answer": refusal_text,
        "declined": bool(declined),
    },
    "evaluation": {
        "semantic_only": {"recall_at_1": base1, "recall_at_5": base5},
        "hybrid_reranked": {"recall_at_1": new1, "recall_at_5": new5},
        "n_queries": len(EVAL),
    },
}

out_path = Path.cwd() / "m2_3_m2_4_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Wrote {out_path}")
print(f"  keyword rank 1 : id {results['keyword_top5'][0]}  (the bug)")
print(f"  RRF rank 1     : id {results['rrf_top5'][0]}")
print(f"  re-ranked top 5: {results['retrieval']['reranked_ids']}")
print(f"  Recall@1       : {base1:.0%} -> {new1:.0%}")
print("\nNow run:  python Day2/autograder/D2_M2.3_M2.4_Hybrid_Rerank_Autograder.py")